# SurfaceEdge — Colab Training

Hybrid CNN + Embedding model for predicting next-day options price changes.

**Before running:** Set Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Dataset will be saved here — update this path if your Drive structure differs
DATASET_ROOT = '/content/drive/My Drive/School/CU Boulder/Semesters/(8) Spring 2026/Neural Nets and Deep Learning/datasets'

import os
os.makedirs(DATASET_ROOT, exist_ok=True)
print(f'Dataset root: {DATASET_ROOT}')

In [ ]:
# ── Step 2: Clone repo and install dependencies ───────────────────────────────
!git clone https://github.com/YOUR_USERNAME/SurfaceEdge.git
%cd SurfaceEdge
!pip install tqdm pandas pyarrow pillow torchvision -q

In [ ]:
# ── Step 3: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'GPU available : {torch.cuda.is_available()}')
print(f'Device        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ── Step 4: Download raw data to Google Drive ─────────────────────────────────
# Creates data/<ticker>/options.parquet and underlying.parquet
# Already-downloaded files are skipped — safe to re-run
import sys
sys.path.insert(0, '/content/SurfaceEdge')

import urllib.request
from pathlib import Path

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}
BASE_URL = "https://static.philippdubach.com/data/options"
DATA_DIR = Path(DATASET_ROOT) / 'data'

from build_dataset import TICKERS

total = len(TICKERS) * 2
completed = 0

for ticker in TICKERS:
    for kind in ('options', 'underlying'):
        completed += 1
        url  = f'{BASE_URL}/{ticker}/{kind}.parquet'
        dest = DATA_DIR / ticker / f'{kind}.parquet'

        if dest.exists() and \
           ((kind == 'options'    and dest.stat().st_size > 1024*1024) or
            (kind == 'underlying' and dest.stat().st_size > 100*1024)):
            print(f'[{completed:>3}/{total}] skipped  {ticker}/{kind}')
            continue

        print(f'[{completed:>3}/{total}] downloading {ticker}/{kind}.parquet ...')
        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req) as r, open(dest, 'wb') as f:
                f.write(r.read())
            print(f'            done ({dest.stat().st_size / 1e6:.1f} MB)')
        except Exception as e:
            print(f'            FAILED — {e}')

print('\nAll downloads complete.')

In [ ]:
# ── Step 5: Build dataset ─────────────────────────────────────────────────────
# Generates surface images and compressed .npz files
# Already-processed days are skipped — safe to re-run
from build_dataset import build
import importlib, build_dataset

DATA_DIR    = str(Path(DATASET_ROOT) / 'data')
DATASET_DIR = str(Path(DATASET_ROOT) / 'dataset')

# Patch paths to point to Drive
build_dataset.DATA_DIR    = Path(DATA_DIR)
build_dataset.DATASET_DIR = Path(DATASET_DIR)
Path(DATASET_DIR).mkdir(parents=True, exist_ok=True)

build()  # all 104 tickers

In [ ]:
# ── Step 6: Train ─────────────────────────────────────────────────────────────
DATASET_DIR = str(Path(DATASET_ROOT) / 'dataset')
SAVE_PATH   = str(Path(DATASET_ROOT) / 'surfaceedge.pt')

# Colab T4 tips:
#   batch_size=1024  — T4 has 15GB VRAM, push it
#   num_workers=2    — Colab can be finicky with high worker counts
#   epochs=20        — watch the MAE curve, stop early if plateauing

!python main.py \
    --dataset      {DATASET_DIR} \
    --epochs       20 \
    --batch_size   1024 \
    --lr           1e-4 \
    --num_workers  2 \
    --save_path    {SAVE_PATH}

In [ ]:
# ── Step 7: Review results ────────────────────────────────────────────────────
checkpoint = torch.load(SAVE_PATH, map_location='cpu')
print(f"Epochs trained  : {checkpoint['epochs_trained']}")
print(f"Final train MAE : {checkpoint['final_train_mae']:.6f}")
print(f"Final test MAE  : {checkpoint['final_test_mae']:.6f}")
print(f"Naive MAE       : {checkpoint['naive_mae']:.6f}")
print(f"Beat naive      : {'YES' if checkpoint['final_test_mae'] < checkpoint['naive_mae'] else 'NO'}")
improvement = (checkpoint['naive_mae'] - checkpoint['final_test_mae']) / checkpoint['naive_mae'] * 100
print(f"Improvement     : {improvement:.2f}% over naive baseline")